This notebook is used to train a classifier that takes as features the ones extracted from Neural networks
and then train a cassifier on the top pf them 

In [11]:
data_folder="../Data/"

import sys
import os

sys.path.append(os.path.abspath("../Codes"))

sys.path.append(os.path.abspath("../_Deep_Learning"))

from Utils import freq_filter, windowize,  RegionWiseStandardizer

from BCI_Library import read_subject, plot_brain, train_models, extract_features_more_bands, saveresults_pickle, select_rows
from BCI_Library import sort_rank, compute_welch_select_regions, extract_features_from_spectra_more_bands,compute_welch
from BCI_Library import plot_hist2d_selected_regions_folds, train_single_model, select_regions_MyCriteria, select_regions_Cohen_effect_size
from BCI_Library import global_selection, build_compiled_mlp

# Selecting regions

# Region Agnostic - test on the same subject

In [31]:
from Deep_Library_BCI import MultiTaskModel
import tensorflow.keras as keras

Tutti_soggetti = [2]

CNN_o_Autoencoder = "Autoencoder" # Autoencoder, # CNN

if CNN_o_Autoencoder == "CNN":
    model = MultiTaskModel(input_shape=(128,1), use_classifier=True, use_decoder=False)
if CNN_o_Autoencoder == "Autoencoder":
    model = MultiTaskModel(input_shape=(128,1), use_classifier=False, use_decoder=True)

model_path_4 ="2026-01-30-18_41_52_Autoencoder_1region_region_agnostic_First_Try" + "/"

model.build(input_shape=(128,1))
model_path_1 = f"../_Deep_Learning/Models_trained/Subject_"
# model path 2 = f"{subject_index}/"
model_path_3 = f"{CNN_o_Autoencoder}" + ("s/" if CNN_o_Autoencoder=="Autoencoder" else "/")

8 128


## Fixed region selection 

In [21]:
Interesting_regions =   [4, 5, 32, 33, 44, 45, 48, 49]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, train_test_split
import tqdm


today = date.today()
sfreq = 250

seed = 224

saveresults = True
filepath_ori = "../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_"

Features_name = CNN_o_Autoencoder + "_extracted"
Region_Selection= "Yes Important Regions - all" # "-"
regions_selected = Interesting_regions

PFR = False

random_seed=224

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

start = 3           # seconds where to start to extract windows
sampling_hz = 250;  start = start*sampling_hz
input_shape = 128   # length of each window
shift = 62          # points to shift for next window in data
num_windows = 11    # how many windows to extract from each trial

end = start + (num_windows-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)


tutte_roi = 68


verbose = False

Classifier_names, meths =  ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier] # ["MLP"], ["MLP"]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}

for DataType in ["EEG"]:
    for Classifier_name, meth in zip(Classifier_names,meths):

        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
       
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm(Tutti_soggetti):

            model_path = model_path_1 + f"{subject_index}/" + model_path_3 + model_path_4 
            Comments = model_path_4[:-1]+ f" Subject addestramento {subject_index}"

            filepath = filepath_ori + f"{subject_index}.pkl"
            file_input = filepath
            file_output = filepath

            data_2_Rest = read_subject(data_folder, DataType, "Baseline",subject_index,verbose=verbose)
            data_2_MI = read_subject(data_folder, DataType, "MI",subject_index,verbose=verbose)
            data = np.concatenate((data_2_Rest, data_2_MI), axis=0)

            data = freq_filter(data, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
            data = data[:,:,start:end]        #  data.shape = (192, rois_selected, 748)

            # y shape: (n_trials,) with negative values for Rest and positive for MI
            y = np.concatenate([-np.ones((data_2_Rest.shape[0])), np.ones((data_2_MI.shape[0]))])

            
            fold_accuracies = []
            for fold_idx, (train_idx, block_idx) in enumerate(kf.split(data, y)):
            ########################## DATA PREPROCESSING ##########################################
            # TODO create (num_windows,0)
                Feat_train_regions_selected = None  #np.empty((len(train_idx)*11,0))
                Feat_test_regions_selected = None # np.empty()

                
                for region_index, regione in enumerate(regions_selected):
                   
                    split_num = fold_idx+1
                    model.load_weights(model_path+f"Split_{split_num}/model.weights.h5")
        

                    X_train = data[train_idx,[regione]]
                    y_train = y[train_idx]
                    # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                    # y_Train shape: (n_trials_train,) ∈ {-1,+1}

                    X_Block = data[block_idx,[regione]]
                    y_Block = y[block_idx]


                    X_val, X_test, y_val, y_test = train_test_split(
                    X_Block, y_Block, test_size=0.5,         
                    shuffle=True, stratify=y_Block,     # preserves label balance
                    random_state=random_seed)

                    y_train = np.repeat(y_train[:,np.newaxis], repeats=1,axis=1)
                    # assegno valore in base a ROI, da ±1 a ±68
                    y_train[:, 0] *= regione+1
                    
                    # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                    # y_Train shape: (n_trials_train, 1_regions) ∈ {-68,...-1,+1,...,+68}

                    y_val = np.repeat(y_val[:,np.newaxis], repeats=1,axis=1)
                    y_val[:, 0] *= regione+1

                    y_test = np.repeat(y_test[:,np.newaxis], repeats=1,axis=1)
                    y_test[:, 0] *= regione+1


                    x_train = X_train.reshape(-1, X_train.shape[-1]) # shape (n_trials*1_regions, n_timepoints)
                    x_val = X_val.reshape(-1, X_val.shape[-1])
                    x_test = X_test.reshape(-1, X_test.shape[-1])

                    y_train = y_train.reshape(-1)
                    y_val = y_val.reshape(-1)
                    y_test = y_test.reshape(-1)

                    # x_train shape: (n_trials_train * 1_regions, n_timepoints)
                    # y_train shape: (n_trials_train * 1_regions,) ∈ {-68,...,-1,+1,...,+68}

                    x_train, y_train = windowize(x_train, y_train, win_len=input_shape, shift=shift)
                    x_val, y_val     = windowize(x_val, y_val, win_len=input_shape, shift=shift)
                    x_test, y_test   = windowize(x_test, y_test, win_len=input_shape, shift=shift)
                    # shape: x -> (n_windows, timepoints) ; y -> (n_windows,)
                    # y is ± region index (SIGN is negative for REST and positive for MI; absolute value is region index)

                
                    #print(f"\n\nx_train shape: {x_train.shape}, x_val shape: {x_val.shape}, x_test shape: {x_test.shape}\n\n")

                    ###################################################################################################################################
                    # NORMALIZATION (per region?)
                    # Optimizing EEG ICA Decomposition with Machine Learning: A CNN-Based Alternative to EEGLAB for Fast and Scalable Brain Activity Analysis
                    # Assessing the Role of EEG Biosignal Preprocessing to Enhance Multiscale Fuzzy Entropy in Alzheimer’s Disease Detection
                    # (when applying on multiple subjects) Cross-Subject EEG-Based Emotion Recognition Through Neural Networks With Stratified Normalization 

                    scaler = RegionWiseStandardizer()
                    x_train = scaler.fit_transform(x_train, y_train)
                    x_val = scaler.transform(x_val, y_val)
                    x_test = scaler.transform(x_test, y_test)
                    # scaler = TraceWiseStandardizer()
                    # x_train = scaler.transform(x_train)
                    # x_val = scaler.transform(x_val)
                    # x_test = scaler.transform(x_test)
                    x_train = x_train[..., np.newaxis]  
                    x_val = x_val[..., np.newaxis]  
                    x_test = x_test[..., np.newaxis]
                    # x_train.shape (n_windows, timepoints, 1)
                    
                    Features_training_1_regione = model.encoder(x_train)
                    Features_test_1_regione = model.encoder(x_test)
                    # Features_training_1_regione  shape (n_windows,16)
                    
                    
                    ################################### Feature Extractions ##########################################
                    if Feat_train_regions_selected is None:
                        Feat_train_regions_selected = np.empty((Features_training_1_regione.shape[0],0))
                        Feat_test_regions_selected = np.empty((Features_test_1_regione.shape[0],0))

                    Feat_train_regions_selected = np.concatenate((Feat_train_regions_selected,Features_training_1_regione),axis = 1) # asse di regione
                    Feat_test_regions_selected = np.concatenate((Feat_test_regions_selected,Features_test_1_regione),axis = 1) # asse di regione


                Feat_All_regions_selected = np.concatenate((Feat_train_regions_selected,Feat_test_regions_selected))
                # Feat_.shape (Trials, selected_ROIs, features for Roi)
                labels_All = np.concatenate(((y_train > 0).astype(int),(y_test > 0).astype(int)))

                train_idx_model = [_ for _ in range(len(Feat_train_regions_selected))]
                val_idx_model = [_ + len(Feat_train_regions_selected) for _ in range(len(Feat_test_regions_selected))]


                if Classifier_name.lower()=="mlp":
                        meth = build_compiled_mlp(input_dim=Feat_All_regions_selected.shape[1])
                        
                acc, cm, model_classifier = train_single_model(Feat_All_regions_selected, labels_All,
                                                    train_idx_model, val_idx_model, method=meth, seed=224, fit_kwargs=MLP_fit_kwargs)

                fold_accuracies.append(acc)
                if verbose:
                    print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                    print(cm)

                #####################################################################################################################

            if verbose:
                print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
                print("Std accuracy:", np.std(fold_accuracies))

            new_row = {
                "ROIs": [regions_selected],
                "NROIs":[len(regions_selected)],
                "DataType": [DataType],
                "freqs_band": ["-"],  
                "subject": [subject_index],
                "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                "Accuracy": [fold_accuracies],
                "Features": [Features_name],
                "Comments": [Comments],
                "Region Selection": [Region_Selection],
                "Date": [today],
                "Personalized_Freq_Range": ["-"],
            }
            Rows=pd.concat((Rows,pd.DataFrame(new_row)))
        

        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG LDA


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:09<00:00,  9.57s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_8.pkl


##############################
 EEG SVC


100%|██████████| 1/1 [00:06<00:00,  6.70s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_8.pkl


##############################
 EEG RandomForest


100%|██████████| 1/1 [00:10<00:00, 10.86s/it]

There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_8.pkl


## Per fold region selection

In [26]:
def compute_power_spectra(data_2_MI, data_2_Rest):
    sfreq = 250
    miff = [8,12]
    maff = [12,30]

    WMI, WRe, FreqMI = compute_welch(data_2_MI, data_2_Rest, sfreq,
                                        kargs_welch={"fmin":miff[0], "fmax":maff[-1]})

    return WMI, WRe


def selezione_cohen(WMI, WRe, train_idx, num_best_ROIs):
    training_index_MI = train_idx[np.where(train_idx<len(WMI))]
    training_index_Re = train_idx[np.where(train_idx>=len(WMI))]-len(WMI)

    regions_selected, effect_size  = select_regions_Cohen_effect_size(wpsdMI=WMI[training_index_MI], 
                                                    wpsdRest=WRe[training_index_MI], 
                                                        important_regions=[], nbest_regions=num_best_ROIs)
    return regions_selected

In [32]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, train_test_split
import tqdm


today = date.today()
sfreq = 250

seed = 224

saveresults = True
filepath_ori = "../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_"

Features_name = CNN_o_Autoencoder + "_extracted"
Region_Selection= "Cohen's d effect size selection" # "-"

PFR = False

random_seed=224

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

start = 3           # seconds where to start to extract windows
sampling_hz = 250;  start = start*sampling_hz
input_shape = 128   # length of each window
shift = 62          # points to shift for next window in data
num_windows = 11    # how many windows to extract from each trial

end = start + (num_windows-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)


num_best_ROIs = 8


verbose = False

Classifier_names, meths =  ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier] # ["MLP"], ["MLP"]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}

for DataType in ["EEG"]:
    for Classifier_name, meth in zip(Classifier_names,meths):

        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
       
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm(Tutti_soggetti):

            model_path = model_path_1 + f"{subject_index}/" + model_path_3 + model_path_4 
            Comments = model_path_4[:-1]+ f" Subject addestramento {subject_index}"

            filepath = filepath_ori + f"{subject_index}.pkl"
            file_input = filepath
            file_output = filepath

            data_2_Rest = read_subject(data_folder, DataType, "Baseline",subject_index,verbose=verbose)
            data_2_MI = read_subject(data_folder, DataType, "MI",subject_index,verbose=verbose)
            data = np.concatenate((data_2_Rest, data_2_MI), axis=0)
            
            WMI, WRe = compute_power_spectra(data_2_MI,data_2_Rest)
            
            data = freq_filter(data, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
            data = data[:,:,start:end]        #  data.shape = (192, rois_selected, 748)

            # y shape: (n_trials,) with negative values for Rest and positive for MI
            y = np.concatenate([-np.ones((data_2_Rest.shape[0])), np.ones((data_2_MI.shape[0]))])

            
            fold_accuracies = []
            for fold_idx, (train_idx, block_idx) in enumerate(kf.split(data, y)):
            ########################## DATA PREPROCESSING ##########################################
            # TODO create (num_windows,0)
                Feat_train_regions_selected = None  #np.empty((len(train_idx)*11,0))
                Feat_test_regions_selected = None # np.empty()
                
                regions_selected = selezione_cohen(WMI, WRe, train_idx, num_best_ROIs)
                
                for region_index, regione in enumerate(regions_selected):
                   
                    split_num = fold_idx+1
                    model.load_weights(model_path+f"Split_{split_num}/model.weights.h5")
        

                    X_train = data[train_idx,[regione]]
                    y_train = y[train_idx]
                    # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                    # y_Train shape: (n_trials_train,) ∈ {-1,+1}

                    X_Block = data[block_idx,[regione]]
                    y_Block = y[block_idx]


                    X_val, X_test, y_val, y_test = train_test_split(
                    X_Block, y_Block, test_size=0.5,         
                    shuffle=True, stratify=y_Block,     # preserves label balance
                    random_state=random_seed)

                    y_train = np.repeat(y_train[:,np.newaxis], repeats=1,axis=1)
                    # assegno valore in base a ROI, da ±1 a ±68
                    y_train[:, 0] *= regione+1
                    
                    # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                    # y_Train shape: (n_trials_train, 1_regions) ∈ {-68,...-1,+1,...,+68}

                    y_val = np.repeat(y_val[:,np.newaxis], repeats=1,axis=1)
                    y_val[:, 0] *= regione+1

                    y_test = np.repeat(y_test[:,np.newaxis], repeats=1,axis=1)
                    y_test[:, 0] *= regione+1


                    x_train = X_train.reshape(-1, X_train.shape[-1]) # shape (n_trials*1_regions, n_timepoints)
                    x_val = X_val.reshape(-1, X_val.shape[-1])
                    x_test = X_test.reshape(-1, X_test.shape[-1])

                    y_train = y_train.reshape(-1)
                    y_val = y_val.reshape(-1)
                    y_test = y_test.reshape(-1)

                    # x_train shape: (n_trials_train * 1_regions, n_timepoints)
                    # y_train shape: (n_trials_train * 1_regions,) ∈ {-68,...,-1,+1,...,+68}

                    x_train, y_train = windowize(x_train, y_train, win_len=input_shape, shift=shift)
                    x_val, y_val     = windowize(x_val, y_val, win_len=input_shape, shift=shift)
                    x_test, y_test   = windowize(x_test, y_test, win_len=input_shape, shift=shift)
                    # shape: x -> (n_windows, timepoints) ; y -> (n_windows,)
                    # y is ± region index (SIGN is negative for REST and positive for MI; absolute value is region index)

                
                    #print(f"\n\nx_train shape: {x_train.shape}, x_val shape: {x_val.shape}, x_test shape: {x_test.shape}\n\n")

                    ###################################################################################################################################
                    # NORMALIZATION (per region?)
                    # Optimizing EEG ICA Decomposition with Machine Learning: A CNN-Based Alternative to EEGLAB for Fast and Scalable Brain Activity Analysis
                    # Assessing the Role of EEG Biosignal Preprocessing to Enhance Multiscale Fuzzy Entropy in Alzheimer’s Disease Detection
                    # (when applying on multiple subjects) Cross-Subject EEG-Based Emotion Recognition Through Neural Networks With Stratified Normalization 

                    scaler = RegionWiseStandardizer()
                    x_train = scaler.fit_transform(x_train, y_train)
                    x_val = scaler.transform(x_val, y_val)
                    x_test = scaler.transform(x_test, y_test)
                    # scaler = TraceWiseStandardizer()
                    # x_train = scaler.transform(x_train)
                    # x_val = scaler.transform(x_val)
                    # x_test = scaler.transform(x_test)
                    x_train = x_train[..., np.newaxis]  
                    x_val = x_val[..., np.newaxis]  
                    x_test = x_test[..., np.newaxis]
                    # x_train.shape (n_windows, timepoints, 1)
                    
                    Features_training_1_regione = model.encoder(x_train)
                    Features_test_1_regione = model.encoder(x_test)
                    # Features_training_1_regione  shape (n_windows,16)
                    
                    
                    ################################### Feature Extractions ##########################################
                    if Feat_train_regions_selected is None:
                        Feat_train_regions_selected = np.empty((Features_training_1_regione.shape[0],0))
                        Feat_test_regions_selected = np.empty((Features_test_1_regione.shape[0],0))

                    Feat_train_regions_selected = np.concatenate((Feat_train_regions_selected,Features_training_1_regione),axis = 1) # asse di regione
                    Feat_test_regions_selected = np.concatenate((Feat_test_regions_selected,Features_test_1_regione),axis = 1) # asse di regione


                Feat_All_regions_selected = np.concatenate((Feat_train_regions_selected,Feat_test_regions_selected))
                # Feat_.shape (Trials, selected_ROIs, features for Roi)
                labels_All = np.concatenate(((y_train > 0).astype(int),(y_test > 0).astype(int)))

                train_idx_model = [_ for _ in range(len(Feat_train_regions_selected))]
                val_idx_model = [_ + len(Feat_train_regions_selected) for _ in range(len(Feat_test_regions_selected))]


                if Classifier_name.lower()=="mlp":
                        meth = build_compiled_mlp(input_dim=Feat_All_regions_selected.shape[1])
                        
                acc, cm, model_classifier = train_single_model(Feat_All_regions_selected, labels_All,
                                                    train_idx_model, val_idx_model, method=meth, seed=224, fit_kwargs=MLP_fit_kwargs)

                fold_accuracies.append(acc)
                if verbose:
                    print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                    print(cm)

                #####################################################################################################################

            if verbose:
                print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
                print("Std accuracy:", np.std(fold_accuracies))

            new_row = {
                "ROIs": [regions_selected],
                "NROIs":[len(regions_selected)],
                "DataType": [DataType],
                "freqs_band": ["-"],  
                "subject": [subject_index],
                "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                "Accuracy": [fold_accuracies],
                "Features": [Features_name],
                "Comments": [Comments],
                "Region Selection": [Region_Selection],
                "Date": [today],
                "Personalized_Freq_Range": ["-"],
            }
            Rows=pd.concat((Rows,pd.DataFrame(new_row)))
        

        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG LDA


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:13<00:00, 13.06s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_2.pkl


##############################
 EEG SVC


100%|██████████| 1/1 [00:12<00:00, 12.15s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_2.pkl


##############################
 EEG RandomForest


100%|██████████| 1/1 [00:18<00:00, 18.68s/it]

There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_2.pkl


In [33]:
pd.read_pickle("../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_2.pkl")

,ROIs,NROIs,DataType,freqs_band,subject,Classifier,Features,Comments,Region Selection,Date,Accuracy,Personalized_Freq_Range
0,"[4, 5, 32, 33, 44, 45, 48, 49]",8.0,EEG,-,2.0,SVC,Autoencoder_extracted,2026-01-30-18_41_52_Autoencoder_1region_region...,Yes Important Regions - all,2026-02-05,"[0.7954545454545454, 0.7363636363636363, 0.712...",-
1,"[4, 5, 32, 33, 44, 45, 48, 49]",8.0,EEG,-,2.0,RandomForest,Autoencoder_extracted,2026-01-30-18_41_52_Autoencoder_1region_region...,Yes Important Regions - all,2026-02-05,"[0.7681818181818182, 0.7227272727272728, 0.698...",-
2,"[4, 5, 32, 33, 44, 45, 48, 49]",8.0,EEG,-,2.0,LDA,Autoencoder_extracted,2026-01-30-18_41_52_Autoencoder_1region_region...,Yes Important Regions - all,2026-02-05,"[0.6409090909090909, 0.6363636363636364, 0.665...",-
3,"[4, 5, 32, 33, 44, 45, 48, 49]",8.0,EEG,-,2.0,SVC,CNN_extracted,2026-02-02-15_59_15_CNN_1region_region_agnosti...,Yes Important Regions - all,2026-02-05,"[0.8090909090909091, 0.7181818181818181, 0.779...",-
4,"[4, 5, 32, 33, 44, 45, 48, 49]",8.0,EEG,-,2.0,RandomForest,CNN_extracted,2026-02-02-15_59_15_CNN_1region_region_agnosti...,Yes Important Regions - all,2026-02-05,"[0.8227272727272728, 0.7363636363636363, 0.746...",-
5,"[4, 5, 32, 33, 44, 45, 48, 49]",8.0,EEG,-,2.0,LDA,CNN_extracted,2026-02-02-15_59_15_CNN_1region_region_agnosti...,Yes Important Regions - all,2026-02-05,"[0.7772727272727272, 0.7045454545454546, 0.789...",-
6,"[4, 5, 32, 33, 44, 45, 48, 49]",8.0,EEG,-,2.0,SVC,CNN_extracted,2026-01-31-13_19_18_CNN_1region_region_agnosti...,Yes Important Regions - all,2026-02-05,"[0.7227272727272728, 0.6818181818181818, 0.741...",-
7,"[4, 5, 32, 33, 44, 45, 48, 49]",8.0,EEG,-,2.0,RandomForest,CNN_extracted,2026-01-31-13_19_18_CNN_1region_region_agnosti...,Yes Important Regions - all,2026-02-05,"[0.7772727272727272, 0.6681818181818182, 0.751...",-
8,"[4, 5, 32, 33, 44, 45, 48, 49]",8.0,EEG,-,2.0,LDA,CNN_extracted,2026-01-31-13_19_18_CNN_1region_region_agnosti...,Yes Important Regions - all,2026-02-05,"[0.740909090909091, 0.6727272727272727, 0.7511...",-
9,"[32, 14, 33, 44, 62, 50, 48, 47]",8.0,EEG,-,2.0,LDA,CNN_extracted,2026-01-31-13_19_18_CNN_1region_region_agnosti...,Cohen's d effect size selection,2026-02-06,"[0.740909090909091, 0.6818181818181818, 0.7703...",-


In [12]:
set(labels_All)

{np.int64(0), np.int64(1)}

In [6]:
file_path

NameError: name 'file_path' is not defined

In [13]:
import pandas as pd
asio = pd.read_pickle( "../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_1.pkl")

# DO NOT DELETE

In [ ]:
# Load model
from Deep_Library_BCI import MultiTaskModel
import tensorflow.keras as keras
model = MultiTaskModel(input_shape=(128,1), use_classifier=False, use_decoder=True)
model.build(input_shape=(128,1))
model.load_weights("../_Deep_Learning/Models_trained/2026-01-30-18_41_52_Autoencoder_1region_region_agnostic_First_Try/Split_1/model.weights.h5")

2026-02-03 12:01:33.529346: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-03 12:01:33.607088: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-03 12:01:35.168641: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1770116496.951234 1977451 gpu_device.cc:2020] Created device /job:lo

8 128


In [ ]:
import numpy as np
zeri = np.zeros(shape=(3,128,1))
model.encoder(zeri)

2026-02-02 16:37:42.890048: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91800


<tf.Tensor: shape=(3, 16), dtype=float32, numpy=
array([[-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542],
       [-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542],
       [-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542]], dtype=float32)>